In [1]:
# Imports
import logging
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import dotenv_values

ws_config = dotenv_values(Path().resolve().parent / ".env.workspace")

# Import custom source code
%load_ext autoreload
%autoreload 2
from virtual_panel import *

# General setup
sns.set_style("darkgrid")
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [2]:
consensus_panel_file = "consensus_panel.feather"
# TODO: standarize the saved gff_path to assembly_tag.gff in get_ncbi_refseq_files.sh
gff_path = f"{ws_config["RAW_DATA_DIR"]}/{ws_config["ASSEMBLY_TAG"]}_genomic.gff"
exon_annotations_path = f"{ws_config["INTERMEDIATE_DATA_DIR"]}/exon_annotations.tsv"

In [3]:
gff_path

'/home/hansu/Remote_Repositories/ws_vcf2clinical/parkinson-ngs-pipeline/panel-bed/data/raw/GCF_000001405.40_GRCh38.p14_genomic.gff'

In [4]:
consensus_panel_df = pd.read_feather(
    f"{ws_config["INTERMEDIATE_DATA_DIR"]}/{consensus_panel_file}"
)
consensus_panel_df.head()

,Name,Status_Consensus,Type,GRCh38_chr,GRCh38_start,GRCh38_end,HGNC_ID,HGNC_symbol,Biotype,Origin
0,ADAR,MIXED,GENE,1,154582062,154627999,HGNC:225,ADAR,protein_coding,AUS
1,ATN1_CAG,MIXED,STR,12,6924463,6942321,HGNC:3033,ATN1,protein_coding,UK
2,ATP13A2,GREEN,GENE,1,16985958,17011928,HGNC:30213,ATP13A2,protein_coding,Both
3,ATP1A3,GREEN,GENE,19,41966582,41997497,HGNC:801,ATP1A3,protein_coding,Both
4,ATP6AP2,MIXED,GENE,X,40579372,40606848,HGNC:18305,ATP6AP2,protein_coding,Both


In [5]:
df = consensus_panel_df
mask = df["Type"] == "GENE"
hgnc_filter = {
    symbol: id
    for symbol, id in zip(df.loc[mask, "HGNC_symbol"], df.loc[mask, "HGNC_ID"])
}
missing = set()

In [6]:
gff_handler = Gff3Handler(set(df[df["Type"] == "GENE"]["HGNC_symbol"]))
annotations_df = gff_handler.parse(gff_path, exon_annotations_path)

In [7]:
# Check genes and exons with unexpected count values
metrics_df = gff_handler.get_metrics()
anomalous_df = metrics_df[
    (metrics_df["gene_lines"] != 1) | (metrics_df["exon_lines"] == 0)
]
display(anomalous_df)
display(consensus_panel_df[consensus_panel_df["HGNC_symbol"].isin(anomalous_df.index)])

,gene_lines,exon_lines
C9orf3,0,0
GBA,0,0


,Name,Status_Consensus,Type,GRCh38_chr,GRCh38_start,GRCh38_end,HGNC_ID,HGNC_symbol,Biotype,Origin
14,C9orf3,MIXED,GENE,9,94726701,95087218,HGNC:1361,C9orf3,protein_coding,AUS
32,GBA,GREEN,GENE,1,155234452,155244699,HGNC:4177,GBA,protein_coding,Both


Reviewing

- C9orf3, [HGNC:1361](https://www.genenames.org/data/gene-symbol-report/#!/hgnc_id/1361): The approved symbol is AOPEP (PUTATIVE).

- GBA [HGNC:4177](https://www.genenames.org/data/gene-symbol-report/#!/hgnc_id/HGNC:4177): the STABLE symbol is GBA1.


In [8]:
# TODO: GffHandler with the sublist
# TODO: Define how append the new results efficiently

In [ ]:
# sketch, comparing if new annotations change

df_merge = (
    consensus_panel_df[consensus_panel_df["Type"] == "GENE"][
        ["HGNC_symbol", "GRCh38_chr", "GRCh38_start", "GRCh38_end"]
    ]
    .copy()
    .merge(annotations_df, how="outer")
)
df_merge["result"] = (df_merge["GRCh38_start"] != df_merge["Start"]) | (
    df_merge["GRCh38_end"] != df_merge["End"]
)
df_merge["result"].value_counts()

result
True     85
False     3
Name: count, dtype: Int64

In [10]:
df_merge[["GRCh38_chr", "RefSeq-Accn", "GRCh38_start", "Start", "GRCh38_end", "End"]]

,GRCh38_chr,RefSeq-Accn,GRCh38_start,Start,GRCh38_end,End
0,1,NC_000001.11,154582062,154582057,154627999,154627997
1,1,NC_000001.11,16985958,16985958,17011928,17011928
2,19,NC_000019.10,41966582,41966582,41997497,41994230
3,X,NC_000023.11,40579372,40580970,40606848,40606848
4,13,NC_000013.11,51930436,51932669,52012125,52012132
...,...,...,...,...,...,...
85,16,NC_000016.10,46656132,46656132,46689518,46689178
86,1,NC_000001.11,119031216,119031216,119140671,119140672
87,X,NC_000023.11,49071470,49074442,49101170,49101178
88,1,NC_000001.11,180632004,180632022,180890251,180890279
